<a href="https://colab.research.google.com/github/ajimotirofiat2-lgtm/wine_quality_regression.ipyn/blob/main/Assg_4_of_ML_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning a Pre-trained CNN on MNIST

This notebook demonstrates how to fine-tune a pre-trained ResNet18 model for the MNIST dataset. It includes steps for data loading, model modification, training, and saving the trained model weights.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

# Define the device to run on
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### 1. Define Hyperparameters

In [ ]:
num_epochs = 2  # As requested, fine-tune for a couple of epochs
batch_size = 64
learning_rate = 0.001
num_classes = 10 # MNIST has 10 classes (digits 0-9)

### 2. Data Loading and Preprocessing

MNIST images are grayscale, but most pre-trained CNNs (like ResNet) expect 3-channel (RGB) input. We will transform the grayscale image to 3 channels by repeating the single channel three times.

In [ ]:
# Define transformations for the MNIST dataset
transform = transforms.Compose([
    transforms.Resize(224), # Resize images to 224x224 for ResNet
    transforms.Grayscale(num_output_channels=3), # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet normalization
])

# Download and load the MNIST training dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Download and load the MNIST test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training data samples: {len(train_dataset)}")
print(f"Test data samples: {len(test_dataset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 504kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.65MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.0MB/s]


Training data samples: 60000
Test data samples: 10000


### 3. Model Definition

We will load a pre-trained ResNet18 model and modify its final fully connected layer to output 10 classes (for MNIST).

In [ ]:
# Load a pre-trained ResNet18 model
model = models.resnet18(pretrained=True)

# Freeze all parameters in the network (optional, but common for fine-tuning)
# for param in model.parameters():
#     param.requires_grad = False

# Get the number of features in the last fully connected layer
num_ftrs = model.fc.in_features

# Replace the last fully connected layer with a new one for MNIST classes
model.fc = nn.Linear(num_ftrs, num_classes)

# Move the model to the specified device
model = model.to(device)

print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### 4. Loss Function and Optimizer

In [ ]:
# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### 5. Training Loop

In [ ]:
print("Starting training...")

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    # Use tqdm for a progress bar
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_accuracy = correct_predictions / total_samples

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

print("Training finished.")

Starting training...


Epoch 1/2: 100%|██████████| 938/938 [04:48<00:00,  3.26it/s]


Epoch [1/2], Loss: 0.0663, Accuracy: 0.9800


Epoch 2/2: 100%|██████████| 938/938 [04:46<00:00,  3.28it/s]

Epoch [2/2], Loss: 0.0357, Accuracy: 0.9892
Training finished.


### 6. Evaluation (Optional)

In [ ]:
print("Starting evaluation...")
model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculation during evaluation
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the {len(test_dataset)} test images: {100 * correct / total:.2f}%')
print("Evaluation finished.")

Starting evaluation...
Accuracy of the model on the 10000 test images: 99.10%
Evaluation finished.


### 7. Save Model Weights

Finally, save the state dictionary of the trained model to a `.pth` file.

In [ ]:
model_save_path = 'mnist_resnet18_finetuned.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model weights saved to {model_save_path}")

Model weights saved to mnist_resnet18_finetuned.pth
